In [1]:
import pandas as pd
import re
import string
import nltk

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:


train_df = pd.read_csv("../data/train_eda.csv")
test_df = pd.read_csv("../data/test_eda.csv")

In [3]:
stop_words = set(stopwords.words('english'))

lemmatizer = WordNetLemmatizer()

In [4]:
def preprocess_text(text):
    
    # Convert to string
    text = str(text)
    
    # Lowercase
    text = text.lower()
    
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Remove punctuation
    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Tokenization
    words = text.split()
    
    # Stopword removal + Lemmatization
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]
    
    return " ".join(words)

In [5]:
train_df["clean_poem"] = train_df["Poem"].apply(preprocess_text)

test_df["clean_poem"] = test_df["Poem"].apply(preprocess_text)

In [6]:
for i in range(3):

    print("ORIGINAL:")
    x = train_df["Poem"].iloc[i]
    print(type(x))
    print(x[:300])

    print("\nCLEANED:")
    y = train_df["clean_poem"].iloc[i]
    print(type(y))
    print(y[:300])

    print("="*50)

ORIGINAL:
<class 'str'>
              In the thick brushthey spend the hottest part of the day,              soaking their hoovesin the trickle of mountain water              the ravine hoardson behalf of the oleander.           

CLEANED:
<class 'str'>
thick brushthey spend hottest part day soaking hoovesin trickle mountain water ravine hoardson behalf oleander
ORIGINAL:
<class 'str'>
   Storms are generous.                                      Something so easy to surrender to, sitting by the window, and then you step out into the garden you were so bored of,             

CLEANED:
<class 'str'>
storm generous something easy surrender sitting window step garden bored
ORIGINAL:
<class 'str'>
 —After Ana Mendieta Did you carry around the matin star? Did you hold forest-fire in one hand? Would you wake to radiate, shimmer, gleam lucero-light? Through the morning would you measure the wingspan of an idea taking off— & by night would

CLEANED:
<class 'str'>
—after ana mendieta carry aroun

In [7]:
raw_vocab = len(
    set(
        " ".join(train_df["Poem"]).lower().split()
    )
)

print(raw_vocab)

11404


In [8]:
clean_vocab = len(
    set(
        " ".join(train_df["clean_poem"]).split()
    )
)

print(clean_vocab)

8842


In [9]:
train_df.to_csv(
    "../data/train_clean.csv",
    index=False
)

test_df.to_csv(
    "../data/test_clean.csv",
    index=False
)

In [10]:
from collections import Counter

for genre in train_df["Genre"].unique():

    text = " ".join(
        train_df[train_df["Genre"] == genre]["clean_poem"]
    ).lower()

    words = text.split()

    print("\n", genre)
    print(Counter(words).most_common(10))


 Music
[('like', 43), ('one', 40), ('say', 34), ('love', 25), ('i’m', 23), ('it’s', 23), ('man', 22), ('time', 20), ('body', 20), ('many', 20)]

 Death
[('one', 38), ('like', 35), ('dead', 30), ('day', 27), ('death', 26), ('know', 23), ('thing', 21), ('white', 21), ('life', 21), ('night', 21)]

 Affection
[('love', 54), ('like', 29), ('come', 18), ('heart', 17), ('eye', 16), ('white', 15), ('said', 15), ('day', 15), ('never', 13), ('one', 13)]

 Environment
[('like', 39), ('tree', 37), ('one', 32), ('night', 29), ('sky', 24), ('sun', 23), ('see', 22), ('white', 21), ('day', 21), ('wind', 20)]


In [11]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(
    stop_words='english',
    ngram_range=(2,2)
)

X = cv.fit_transform(train_df["clean_poem"])

freq = X.sum(axis=0).A1

ngrams = zip(cv.get_feature_names_out(), freq)

sorted(ngrams, key=lambda x: x[1], reverse=True)[:20]

[('don know', np.int64(7)),
 ('ah yes', np.int64(6)),
 ('parking lot', np.int64(6)),
 ('died august', np.int64(5)),
 ('night come', np.int64(5)),
 ('yes ah', np.int64(5)),
 ('best heart', np.int64(4)),
 ('blue face', np.int64(4)),
 ('dead man', np.int64(4)),
 ('flower long', np.int64(4)),
 ('footstep coming', np.int64(4)),
 ('long ago', np.int64(4)),
 ('look like', np.int64(4)),
 ('love porgy', np.int64(4)),
 ('make think', np.int64(4)),
 ('pulse heart', np.int64(4)),
 ('river like', np.int64(4)),
 ('ancient symbolswe', np.int64(3)),
 ('apple tree', np.int64(3)),
 ('attend attendwait', np.int64(3))]